# Workflow Agents

Orchestrate sub-agents with a SequentialAgent.

> Part of the **ADK Katas**. Work top-to-bottom: read the concept, fill in the
> `# TODO`s in the starter cell, then run the **Check** cell — it grades your
> code offline (no API call). The **Run it live** cell needs a `GOOGLE_API_KEY`.


## Kata 08 — Workflow Agents

Not everything should be left to the LLM to route. **Workflow agents** run
sub-agents on a fixed control-flow:

- `SequentialAgent` — run sub-agents one after another (a pipeline)
- `ParallelAgent` — run them concurrently
- `LoopAgent` — repeat until a condition / max iterations

They're deterministic orchestration, no model call of their own.

## Your task

Build a `SequentialAgent` named `"writer_pipeline"` whose `sub_agents` are,
in order: a `drafter` agent then a `reviewer` agent. Bind it to `root_agent`.

In [ ]:
# Setup — run me first
import sys, pathlib
# Make the kata library importable whether opened from adk-katas/ or adk-katas/notebooks/
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "kata_helpers.py").exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from kata_helpers import *          # load_env, has_api_key, run_agent, check, grade, requires_key
from kata_content import BY_SLUG

KATA = BY_SLUG["workflow-agents"]
load_env()
print("API key configured:" , has_api_key())


In [ ]:
# ✏️ Your code — fill in the TODOs
from google.adk.agents import Agent, SequentialAgent

M = "gemini-2.5-flash"

# TODO: drafter — writes a first draft from the user's topic.
drafter = None
# TODO: reviewer — improves the draft.
reviewer = None

# TODO: root_agent = SequentialAgent("writer_pipeline", sub_agents=[drafter, reviewer])
root_agent = None

In [ ]:
# ✅ Check your work (offline — grades the symbols you defined above)
results = KATA.run_checks(globals())
grade([check(r.label, r.passed, r.detail) for r in results])


In [ ]:
# ▶️ Run it live (needs GOOGLE_API_KEY). Each run is a fresh single-turn session.
agent = globals().get(KATA.chat_symbol)

if not KATA.chattable:
    print("This kata has no chat agent — the Check cell is the goal. 🎯")
elif has_api_key() and agent is not None:
    result = await run_agent(agent, 'Topic: a lighthouse in winter.')   # noqa: F704  (top-level await is fine in Jupyter)
    print("Response:", result.text)
    if result.tool_calls:
        print("Tools called:", result.tool_calls, result.tool_args)
    if result.transfers:
        print("Transferred to:", result.transfers)
    if result.state:
        print("Session state:", result.state)
else:
    requires_key(lambda: None)


---
### Stuck? Reveal the reference solution

<details>
<summary>Show solution</summary>

```python
from google.adk.agents import Agent, SequentialAgent

M = "gemini-2.5-flash"

drafter = Agent(
    name="drafter", model=M,
    instruction="Write a short first draft about the user's topic.",
    output_key="draft",
)
reviewer = Agent(
    name="reviewer", model=M,
    instruction="Improve the draft in state['draft']; return the final text.",
)
root_agent = SequentialAgent(
    name="writer_pipeline",
    sub_agents=[drafter, reviewer],
)
```

</details>

When you're done, try the same kata in the React app's live chat (`./dev.sh`
from the repo root) to watch the tool-call traces.
